In [1]:
#1 Installation of Libraries
!pip install -q -U \
transformers\
datasets\
accelerate\
peft\
trl\
bitsandbytes\
huggingface_hub\
sentencepiece



In [2]:
#2 Import Library
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)


from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig     #TRL 1.x me TrainingArguments ki jagah ya uske extension ke roop me use hota hai.


In [3]:
#3 check gpu
import torch

print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

Torch : 2.11.0+cu128
CUDA  : True
GPU   : Tesla T4


In [4]:
#4 load dataset

dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split = "train")

# small sample for testing
train_dataset = dataset

print(train_dataset)


Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})


In [5]:
#5 load tokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code = True
)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer Loaded Successfully ✅")
print("Tokenizer Vocabulary Size :", tokenizer.vocab_size)
print("Pad Token :", tokenizer.pad_token)
print("EOS Token :", tokenizer.eos_token)

Tokenizer Loaded Successfully ✅
Tokenizer Vocabulary Size : 151643
Pad Token : <|im_end|>
EOS Token : <|im_end|>


In [6]:
#6 formate dataset

def formatting_func(example):

     # Instruction ko extract kiya
    instruction = example["instruction"]

      # Context ko extract kiya
    context = example["context"]

    # Response ko extract kiya
    response = example["response"]

    # Agar context available hai
    if context:

        user_prompt = (
            f"{instruction}\n\n"
            f"Context:\n{context}"
        )

    # Agar context empty hai
    else:

        user_prompt = instruction

    # Chat messages banaye
    messages = [
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": response
        }
    ]

    # Qwen chat template apply kiya
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


# Dataset formatting apply karo
train_dataset = train_dataset.map(formatting_func)

print(train_dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
When did Virgin Australia start operating?

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>
<|im_start|>assistant
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>



In [7]:
#7 Keep only the formatted text column

train_dataset = train_dataset.remove_columns(
    ["instruction", "context", "response", "category"]
)

print(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['text'],
    num_rows: 15011
})
{'text': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhen did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>\n<|im_start|>assistant\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>\n"}


In [8]:
#8 4-bit quqntization

bnb_config = BitsAndBytesConfig(

    # Model ko 4-bit me load karna hai
    load_in_4bit = True,

     # Quantization type
    bnb_4bit_quant_type = "nf4",

    # Computation kis datatype me hogi
    bnb_4bit_compute_dtype = torch.float16,

    # Double Quantization enable
    bnb_4bit_use_double_quant = True
)



In [9]:
#9 load model

model = AutoModelForCausalLM.from_pretrained(
    model_name,

    # Apply 4-bit quantization
    quantization_config = bnb_config,

    device_map = "auto",    # Automatically use GPU

    # Reduce CPU RAM usage while loading
    low_cpu_mem_usage=True,

    # Trust model's custom code
    trust_remote_code=True,

    torch_dtype=torch.float16,

)

# Disable KV Cache during training
model.config.use_cache = False


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [10]:
#10 Prepare Model for QLoRA Training

model = prepare_model_for_kbit_training(model)

print("Model is Ready for QLoRA Training ✅")

Model is Ready for QLoRA Training ✅


In [11]:
#11 LoRA Config

lora_config = LoraConfig(
    r =16,                           # Adapter Size

    # LoRA Scaling factor
    lora_alpha = 32,                 #Adapter Strength

     # Dropout for LoRA layers
    lora_dropout = 0.05,             #Prevent Overfitting

     # Don't train bias parameters
    bias = "none",

    # Model Task type
    task_type = "CAUSAL_LM",

    # Layers where LoRA will be applied
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

    ]
)

# Apply LoRA Adapter to the Model
model = get_peft_model(    # yhi line base model ko Lora model bnati ,without this normal model hi rhega
    model,
    lora_config
)


# Convert only trainable LoRA weights to float32
model = model.to(torch.float32)


# Show trainable parameters
model.print_trainable_parameters()

# Verify Trainable Parameter dtype
print("Trainable Parameter Dtype:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Trainable Parameter Dtype:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32


In [12]:
#12 Training Arguments



training_args = TrainingArguments(

    # Output directory
    output_dir="./results",

    # Number of training epochs
    num_train_epochs=2,

    # Batch size per GPU
    per_device_train_batch_size=4,   # Ek baar me GPU kitne examples dekhegi

    # Gradient accumulation
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=2e-4,

    # Optimizer
    optim="paged_adamw_8bit",

    # Weight decay to reduce overfitting
    weight_decay=0.01,

    # Save checkpoint every epoch
    save_strategy="epoch",          #har epoch k baad checkpoint save krega Agar training beech me ruk jaye to wahi se continue kar sakte ho.

    # Log after every 10 steps
    logging_steps=10, # Har 10 training steps ke baad loss print karega

    logging_strategy="steps",

    # LR Scheduler
    lr_scheduler_type="cosine",

    # Warmup
    warmup_ratio=0.03,      # Starting me learning rate dheere-dheere increase karega.

    # Disable bf16
    bf16=False,
    tf32=False,
    fp16=False,


    # Keep only the latest 2 checkpoints
    save_total_limit=2,

    # Random seed
    seed=42,

    # Report to WandB/TensorBoard
    report_to="none",

)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
#13 Trainer

trainer = SFTTrainer(

    # Base Model
    model=model,

    # Training Dataset
    train_dataset=train_dataset,

    # Tokenizer
    processing_class=tokenizer,

    # Training Arguments
    args=training_args,

    # # LoRA Configuration
    # peft_config=lora_config,  Agar get_peft_model() pehle call kiya hai → peft_config mat do.
    #Agar base model pass kar rahe ho aur trainer se LoRA apply karwana hai → peft_config do.
)






[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16


In [15]:
#12 Start Fine-Tuning
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.910648
20,2.541368
30,2.356953
40,2.206572
50,2.067128
60,1.975702
70,2.023490
80,1.967326
90,1.957447
100,2.033451


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=1878, training_loss=1.9188201044060298, metrics={'train_runtime': 7366.597, 'train_samples_per_second': 4.075, 'train_steps_per_second': 0.255, 'total_flos': 2.57746880833728e+16, 'train_loss': 1.9188201044060298, 'entropy': 1.8856155954558274, 'num_tokens': 6014298.0, 'mean_token_accuracy': 0.6067734895081356, 'epoch': 2.0})

# New Section

In [16]:
#13 Save the Model
trainer.save_model("./qwen_dolly_Lora")    # ye bs lora adapter save krta h base model nhi usko dobara load krna hoga huggingFace se
tokenizer.save_pretrained("./qwen_dolly_Lora")
print("Model Saved Successfully ✅")

Model Saved Successfully ✅


In [22]:
#14 Final Model inference (testing)


def ask_model(question, context=""):

    # Set model to evaluation mode
    model.eval()

    # Prepare user prompt
    if context.strip():
        user_prompt = f"{question}\n\nContext:\n{context}"
    else:
        user_prompt = question

    # Create chat messages
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Apply Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert prompt into input tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Disable gradient calculation during inference
    with torch.no_grad():

        # Generate model response
        outputs = model.generate(

            **inputs,

            # Maximum number of new tokens to generate
            max_new_tokens=200,

            # Lower temperature gives more deterministic answers
            temperature=0.2,

            # Disable random sampling
            do_sample=False,

            # Reduce repeated text generation
            repetition_penalty=1.1,

            # Padding token
            pad_token_id=tokenizer.eos_token_id
        )

    # Convert output tokens back to text
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Print generated response
    print(response)



In [23]:
ask_model("What is Machine Learning?")


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is Machine Learning?
assistant
Machine learning is the study of algorithms and statistical models that enable computers to learn from data and improve their performance on tasks without being explicitly programmed. It has been applied in many fields including natural language processing, computer vision, recommendation systems, and autonomous vehicles.


In [25]:
ask_model("Summarize the following paragraph -Artificial Intelligence is a branch of computer science...")

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Summarize the following paragraph -Artificial Intelligence is a branch of computer science...
assistant
Artificial intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence. AI systems learn from data and improve over time through trial and error. They have been used in a wide range of applications, including language translation, image recognition, autonomous vehicles, and natural language processing.


In [26]:
ask_model("Write a professional leave email.")

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a professional leave email.
assistant
Subject: Leave Request

Dear [Employee Name],

I am writing to request an additional day off from work due to unforeseen circumstances. I have been working hard and have been able to meet all of my deadlines on time.

However, I am currently facing some unexpected challenges that require me to take a break for a few days. I would like to be able to return to work as soon as possible, but unfortunately, this is not possible at the moment.

Please let me know if you need any further information or assistance in arranging a leave. I will be available during your availability times to discuss any concerns you may have.

Thank you for considering my request. I look forward to hearing back from you soon.

Best regards,

[Your Name]


In [27]:
ask_model("Write a Python function to check whether a number is prime.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to check whether a number is prime.
assistant
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

print(is_prime(7)) # prints: True
print(is_prime(8)) # prints: False
print(is_prime(13)) # prints: True
print(is_prime(14)) # prints: False
print(is_prime(15)) # prints: False
print(is_prime(16)) # prints: False
print(is_prime(17)) # prints: True


In [28]:
ask_model("If a train travels 60 km in one hour, how long will it take to travel 150 km?")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
If a train travels 60 km in one hour, how long will it take to travel 150 km?
assistant
To calculate the time required for a train traveling at a speed of 60 km/hour to cover 150 km, we can use the formula: 
Time = Distance / Speed

Substituting the values:
Time = 150 km / 60 km/hour
Time = 2.5 hours

Therefore, it would take 2.5 hours or 150 minutes to travel 150 km at a speed of 60 km/hour.


In [18]:
#15 Save final PEFT adapter locally

model.config.use_cache = True


save_path = "./qwen-dolly-qlora"


model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)


print("Model saved successfully ✅")

Model saved successfully ✅


In [19]:
#16 — Check Saved Files

import os

print(os.listdir("./qwen-dolly-qlora"))

['chat_template.jinja', 'README.md', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [21]:
question = "What is Machine Learning?"
context = ""

In [30]:
#17 Login to Hugging Face

from huggingface_hub import login
login()





In [31]:
#18 push LoRA Adapter to huggingface

repo_name = "Shivanisaini/qwen-dolly-qlora"

model.push_to_hub(repo_name)

tokenizer.push_to_hub(repo_name)


print("Uploaded to Hugging Face ✅")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  14%|#4        |  612kB / 4.35MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpmwkihb0z/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Uploaded to Hugging Face ✅


In [32]:
#19 — Create Backup Zip
import shutil

shutil.make_archive(
    "qwen-dolly-qlora",
    "zip",
    "./qwen-dolly-qlora"
)


print("Zip created ✅")

Zip created ✅


In [33]:
#20 Download Model (Backup)
from google.colab import files

files.download("qwen-dolly-qlora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
# Merge LoRA into Base Model

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()

print("LoRA Merged Successfully ✅")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


LoRA Merged Successfully ✅


In [37]:
# Save Merged Model


# Enable KV cache for inference
merged_model.config.use_cache = True

# Folder to save merged model
save_path = "./qwen-dolly-merged"

# Save merged model
merged_model.save_pretrained(
    save_path,
    safe_serialization=True
)

# Save tokenizer
tokenizer.save_pretrained(save_path)

print("Merged Model Saved Successfully ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged Model Saved Successfully ✅


In [38]:
# Test Merged Model


merged_model.eval()

messages = [
    {
        "role": "user",
        "content": "What is Machine Learning?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(merged_model.device)

with torch.no_grad():

    outputs = merged_model.generate(

        **inputs,

        max_new_tokens=200,

        temperature=0.2,

        do_sample=False,

        repetition_penalty=1.1,

        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is Machine Learning?
assistant
Machine learning is a subfield of artificial intelligence that involves the design and development of algorithms for making decisions under uncertainty. It aims to enable computers to learn from data without being explicitly programmed or trained on specific tasks. This process can be used in various applications such as image recognition, natural language processing, recommendation systems, and more. The goal is to create intelligent machines that can adapt and improve their performance over time based on experience and feedback.


In [39]:
# Upload Merged Model


repo_name = "Shivanisaini/qwen-dolly-merged"

merged_model.push_to_hub(repo_name)

tokenizer.push_to_hub(repo_name)

print("Merged Model Uploaded Successfully ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...w3ilipi/model.safetensors:   2%|2         | 15.4MB /  730MB            

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpik3s32sp/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Merged Model Uploaded Successfully ✅


In [40]:
# Download Merged Model


import shutil
from google.colab import files

shutil.make_archive(
    "qwen-dolly-merged",
    "zip",
    "./qwen-dolly-merged"
)

files.download("qwen-dolly-merged.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>